# Analyse II: Alignierte Embeddings

Nächste Nachbarn der Zielwörter in den alignierten Zeitscheiben und ihre Veränderung über die Zeit.

In [ ]:
import polars as pl

from zeitenwende_sgns.config import ALIGNED_DIR
from zeitenwende_sgns.neighbors import (
    load_slices,
    neighbors_over_time,
    plot_similarity_matrices,
    similarity_matrix,
)
from zeitenwende_sgns.targets import load_targets

pl.Config.set_tbl_rows(30)

SOURCE = "cpp_bt"
PERIOD = "year"

slices = load_slices(ALIGNED_DIR / SOURCE / PERIOD)
targets = load_targets()
list(slices)

## Abdeckung der Zielwörter

In wie vielen Zeitscheiben jedes Zielwort im Vokabular ist (Mindestfrequenz beim Training).

In [ ]:
coverage = pl.DataFrame(
    [
        {
            "target": target,
            **{label: target in wv.key_to_index for label, wv in slices.items()},
        }
        for target in targets
    ]
).with_columns(slices=pl.sum_horizontal(pl.exclude("target")))

coverage["slices"].value_counts().sort("slices", descending=True)

In [ ]:
coverage.filter(pl.col("slices") < len(slices)).sort("slices")

## Nachbarn je Zeitscheibe

In [ ]:
WORD = "Angriffskrieg"

neighbors_over_time(slices, WORD, n=10).pivot("label", index="rank", values="neighbor")

## Diachrone Ähnlichkeitsmatrizen

Nach Kiyama et al. (2025): Kosinus-Ähnlichkeit der Vektoren eines Wortes zwischen allen Paaren von Zeitscheiben.
Blöcke hoher Ähnlichkeit zeigen stabile Phasen, ein Wechsel zwischen Blöcken eine Bedeutungsverschiebung, 
einzelne helle Zeilen/Spalten kurzfristige Ausschläge. Zeitscheiben ohne das Wort bleiben leer.

In [ ]:
WORDS = [
    "Zeitenwende",
    "Russland",
    "Bundesregierung",
    "Krieg",
    "Waffenlieferung",
    "Bundeswehr",
    "Aufrüstung",
    "Benzin",
    "Haus",
    "Wasser",
    "Prozent",
]

plot_similarity_matrices(slices, WORDS, scale="word", cmap="inferno_r");

In [ ]:
WORD = "Prozent"
neighbors_over_time(slices, WORD, n=10).pivot("label", index="rank", values="neighbor")
